### SetUp

In [1]:
from project_modules.project_imports import *
from pathlib import Path

from project_modules.classes import Patient
from project_modules.classes import Clinic

from project_modules.rules import call_a_rule

from project_modules.simulation_tools import create_appointments, plot_line_graph
from project_modules.simulation_tools import establish_attendance, random_patient_sample
from project_modules.simulation_tools import get_margin_errors, check_convergence_mean, confidence_interval, calculate_summary

from openpyxl.utils import get_column_letter

In [2]:
project_path = Path.cwd()
print(project_path)

/workspaces/fair-ml-overbooking


In [3]:
def safe_divide(numerator, denominator):
    return numerator / denominator.replace(0, np.nan)

def confidence_bounds(mean_value, margin):
    return mean_value - margin, mean_value + margin

def run_metric_tests(series_a, series_b, metric_name):
    """
    Runs 4 t-tests:
        greater / two-sided
        welch correction True / False
    Returns dataframe.
    """
    tests = []

    configs = [
        ("greater", True),
        ("greater", False),
        ("two-sided", True),
        ("two-sided", False),
    ]

    for alternative, correction in configs:

        df = ttest(
            series_a,
            series_b,
            correction=correction,
            alternative=alternative,
            confidence=0.95
        )

        tests.append({
            "metric": metric_name,
            "alternative": alternative,
            "welch_correction": correction,
            "t_stat": df["T"].values[0],
            "p_value": df["p_val"].values[0]
        })

    return pd.DataFrame(tests)

def autosize_excel_columns(writer, sheet_name):
    ws = writer.sheets[sheet_name]

    for col_cells in ws.columns:
        max_len = max(
            len(str(cell.value)) if cell.value is not None else 0
            for cell in col_cells
        )
        col_letter = get_column_letter(col_cells[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 2, 12)

### Parameter definition

In [4]:
# ============================================================
# #                   CLINIC CONFIGURATION                   #
# ============================================================

# Configuration Constants
SEED = 42
SLOT_DURATION_MINS = 20
NUM_SERVERS = 1
OPERATING_HOURS = 10
SIMULATION_DAYS = 7
EXTRA_DEMAND_RATE = 0.35 # The percentage of extra patients to simulate beyond capacity

# Derived calculations
SLOTS_PER_DAY = (60 // SLOT_DURATION_MINS) * OPERATING_HOURS
TOTAL_SLOT_CAPACITY = SLOTS_PER_DAY * NUM_SERVERS * SIMULATION_DAYS
TOTAL_PATIENT_POOL = int(TOTAL_SLOT_CAPACITY * (1 + EXTRA_DEMAND_RATE))

print(f"Slots per day: {SLOTS_PER_DAY}")
print(f"Total slot capacity per day: {TOTAL_SLOT_CAPACITY}")
print(f"Total patient pool for simulation: {TOTAL_PATIENT_POOL}")


Slots per day: 30
Total slot capacity per day: 210
Total patient pool for simulation: 283


In [5]:
# ============================================================
# #                  OVERBOOKING AND POLICY                  #
# ============================================================

RULE_NAME = "simple_pairing"
OVERBOOKING_LEVEL = 10

In [6]:
# ============================================================
# #                  SIMULATION PARAMETERS                   #
# ============================================================

PROTECTED_GROUP_PROPORTION = 0.14
NUM_REPLICAS = 30
NUM_ITERATIONS = 30

THRESHOLDS = [(0.1, 0.1), (0.15, 0.15), (0.2, 0.2), (0.25, 0.25)]

# Print thresholds
print("Thresholds for evaluation:")
for idx, (thr_1, thr_2) in enumerate(THRESHOLDS):
    print(f"Thresholds {idx + 1}: Protected = {thr_1}, Unprotected = {thr_2}")

Thresholds for evaluation:
Thresholds 1: Protected = 0.1, Unprotected = 0.1
Thresholds 2: Protected = 0.15, Unprotected = 0.15
Thresholds 3: Protected = 0.2, Unprotected = 0.2
Thresholds 4: Protected = 0.25, Unprotected = 0.25


### Initialize patients and load model probabilities

In [7]:
X_file_path = project_path / "prediction_model/X.npy"
y_file_path = project_path / "prediction_model/y.npy"
column_file_path =  project_path / "prediction_model/column_names.pkl"

with open(column_file_path, 'rb') as file:
    column_names = pickle.load(file)

X = np.load(X_file_path, allow_pickle=True)
y = np.load(y_file_path, allow_pickle=True)

X = pd.DataFrame(X)
X.columns = column_names

patients_original = [Patient(i, **row) for i, row in enumerate(X.to_dict(orient='records'))]

In [8]:
patient_proba_path = project_path / "patients_with_proba.pkl"

with open(patient_proba_path, 'rb') as f:
    patients_with_proba = pickle.load(f)

patients = patients_with_proba.copy()
probas = [patient.proba for patient in patients]

ppv_npv_results_file_path = "prediction_model/ppv_npv_results.xlsx"
ppv_npv_df = pd.read_excel(ppv_npv_results_file_path)

In [9]:
# Splitting patients into cohorts
protected_group = [p for p in patients if p.protected]
general_group = [p for p in patients if not p.protected]

# Calculating requirements based on constants
required_protected = int(TOTAL_PATIENT_POOL * PROTECTED_GROUP_PROPORTION)
required_general = TOTAL_PATIENT_POOL - required_protected

# Validation Checks
is_protected_pool_sufficient = len(protected_group) >= required_protected

print(f"--- Population Summary ---")
print(f"Total Available Patients: {len(patients)}")
print(f"Protected Group Size: {len(protected_group)}")
print(f"General Group Size: {len(general_group)}")
print(f"--- Sampling Requirements ---")
print(f"Target Sample Size: {TOTAL_PATIENT_POOL}")
print(f"Protected Patients Needed: {required_protected}")
print(f"General Patients Needed: {required_general}")
print(f"Sufficient Protected Patients: {is_protected_pool_sufficient}")

--- Population Summary ---
Total Available Patients: 104176
Protected Group Size: 11291
General Group Size: 92885
--- Sampling Requirements ---
Target Sample Size: 283
Protected Patients Needed: 39
General Patients Needed: 244
Sufficient Protected Patients: True


In [10]:
prot_probas = [p.proba for p in patients if p.protected]
nonprot_probas = [p.proba for p in patients if not p.protected]

print(f"Protected proba: mean={np.mean(prot_probas):.4f}, std={np.std(prot_probas):.4f}")
print(f"NonProt proba:   mean={np.mean(nonprot_probas):.4f}, std={np.std(nonprot_probas):.4f}")

# Distribution comparison
from scipy.stats import ks_2samp
stat, pval = ks_2samp(prot_probas, nonprot_probas)
print(f"KS test: statistic={stat:.4f}, p-value={pval:.4g}")

Protected proba: mean=0.1128, std=0.0655
NonProt proba:   mean=0.1393, std=0.1579
KS test: statistic=0.1071, p-value=4.345e-101


### Simulation

In [11]:
files_prefix = f"{RULE_NAME}_overbooking{OVERBOOKING_LEVEL}_sample{TOTAL_PATIENT_POOL}_prot{PROTECTED_GROUP_PROPORTION:.2f}"

DEBUG_VERBOSE = False
model = None

results_summary_rows = []

metric_rows = {
    "PatientWT":     [],
    "ConflictRate":  [],
    "IdleTime":      [],
    "OverTime":      [],
    "OverallWT":     [],
    "TotalAttended": [],
}

for threshold_iter, thresholds in enumerate(THRESHOLDS):

    clear_output(wait=True)

    threshold_protected = thresholds[0]
    threshold_no_protected = thresholds[1]

    protected_ppv = ppv_npv_df[ppv_npv_df["threshold"] == threshold_protected]["protected_ppv"].values[0]
    non_protected_ppv = ppv_npv_df[ppv_npv_df["threshold"] == threshold_no_protected]["unprotected_ppv"].values[0]
    protected_npv = ppv_npv_df[ppv_npv_df["threshold"] == threshold_protected]["protected_npv"].values[0]
    non_protected_npv = ppv_npv_df[ppv_npv_df["threshold"] == threshold_no_protected]["unprotected_npv"].values[0]

    with open("patients_with_proba.pkl", "rb") as f:
        patients_og = pickle.load(f)

    all_measures = []
    convergence_values = []

    print(
        f"Simulating PT {threshold_protected} & NPT {threshold_no_protected} | "
        f"[P PPV {protected_ppv:.4f}] [NP PPV {non_protected_ppv:.4f}] "
        f"[P NPV {protected_npv:.4f}] [NP NPV {non_protected_npv:.4f}]"
    )

    # ============================================================
    # MONTE CARLO LOOP
    # ============================================================
    convergence = False
    iterations  = 0

    while not convergence and iterations < NUM_REPLICAS:

        start = time.time()
        iterations += 1

        patients = copy.deepcopy(patients_og)
        created_appointments = create_appointments(NUM_SERVERS, SIMULATION_DAYS, OPERATING_HOURS, SLOT_DURATION_MINS)

        sampled_random_patient_list = random_patient_sample(patients, TOTAL_PATIENT_POOL, PROTECTED_GROUP_PROPORTION, SIMULATION_DAYS)

        appointments_from_rule, num_refused = call_a_rule(
            sampled_random_patient_list, created_appointments,
            RULE_NAME, model,
            threshold_protected, threshold_no_protected,
            overbooking_level=OVERBOOKING_LEVEL,
        )

        for inner_iter in range(NUM_ITERATIONS):

            appointments = copy.deepcopy(appointments_from_rule)
            random_patient_list = copy.deepcopy(sampled_random_patient_list)

            attendance_random_patient_list = establish_attendance(
                random_patient_list,
                protected_ppv, non_protected_ppv, protected_npv, non_protected_npv,
                threshold_protected, threshold_no_protected,
            )

            clinic = Clinic(attendance_random_patient_list, appointments, SLOT_DURATION_MINS)
            clinic.simulation()

            measures = clinic.get_measures()

            # Flatten per-server lists into single scalars for easier aggregation later.
            measures["idle_time_total"] = float(sum(measures["idle_time_server"]))
            measures["over_time_total"] = float(sum(measures["over_time"]))

            all_measures.append(measures)

            del appointments, random_patient_list, attendance_random_patient_list, clinic

        if iterations >= 20:

            tmp_df = pd.DataFrame(all_measures).copy()
            tmp_df["replica_id"] = np.repeat(np.arange(iterations), NUM_ITERATIONS)[:len(tmp_df)]

            replica_records = (
                tmp_df.drop(columns=["idle_time_server", "over_time"], errors="ignore")
                .groupby("replica_id")
                .mean(numeric_only=True)
                .to_dict(orient="records")
            )

            convergence, diff = check_convergence_mean(replica_records, converge_mean=0.02, verbose=DEBUG_VERBOSE)
            convergence_values.append(diff)

        end = time.time()
        print(f"Ran Successfully Replica {iterations}/{NUM_REPLICAS} in {end-start:.2f}s")

        del created_appointments, patients

    print("* * * Ran Successfully All Replicas!!!")

    measures_df = pd.DataFrame(all_measures)
    measures_df["replica_id"] = np.repeat(np.arange(iterations), NUM_ITERATIONS)[:len(measures_df)]

    prot_anchor = sum(1 for p in sampled_random_patient_list if p.protected and not p.overbooked_target and p.assigned)
    nonprot_anchor = sum(1 for p in sampled_random_patient_list if not p.protected and not p.overbooked_target and p.assigned)
    prot_flagged = sum(1 for p in sampled_random_patient_list if p.protected and p.overbooked_target and p.assigned)
    nonprot_flagged = sum(1 for p in sampled_random_patient_list if not p.protected and p.overbooked_target and p.assigned)

    prot_total = prot_anchor + prot_flagged
    nonprot_total = nonprot_anchor + nonprot_flagged
    prot_flagged_pct = (prot_flagged / prot_total) if prot_total > 0 else 0
    nonprot_flagged_pct = (nonprot_flagged / nonprot_total) if nonprot_total > 0 else 0

    replica_grouped = (
        measures_df
        .groupby("replica_id")
        .agg(
            prot_total_wt = ("clients_total_waiting_time protected class", "sum"),
            nonprot_total_wt = ("clients_total_waiting_time non protected class", "sum"),
            prot_attend = ("protected_assistance", "sum"),
            nonprot_attend = ("non_protected_assistance", "sum"),
            conflict_prot = ("conflict_slots_protected", "sum"),
            conflict_nonprot = ("conflict_slots_non_protected", "sum"),
            total_wt = ("total_waiting_time", "sum"),
            total_attended = ("total_attended_patients", "sum"),
            idle_time = ("idle_time_total", "sum"),
            over_time = ("over_time_total", "sum"),
        )
        .reset_index()
    )

    # Operational metrics — daily averages per realization
    idle_time_replica = replica_grouped["idle_time"] / (NUM_ITERATIONS * SIMULATION_DAYS)
    over_time_replica = replica_grouped["over_time"] / (NUM_ITERATIONS * SIMULATION_DAYS)
    total_attended_replica = replica_grouped["total_attended"] / (NUM_ITERATIONS * SIMULATION_DAYS)

    # Overall patient WT — ratio of sums (correct aggregation)
    overall_patient_wt_replica = safe_divide(replica_grouped["total_wt"], replica_grouped["total_attended"]).dropna()

    # Per-group mean WT — headline fairness metric
    prot_mean_wt_replica = safe_divide(replica_grouped["prot_total_wt"], replica_grouped["prot_attend"]).dropna()
    nonprot_mean_wt_replica = safe_divide(replica_grouped["nonprot_total_wt"], replica_grouped["nonprot_attend"]).dropna()

    # Per-group conflict rate — mechanism metric
    conflict_rate_prot_replica = safe_divide(replica_grouped["conflict_prot"], replica_grouped["prot_attend"]).dropna()
    conflict_rate_nonprot_replica = safe_divide(replica_grouped["conflict_nonprot"], replica_grouped["nonprot_attend"]).dropna()

    # Diagnostic: track effective sample size after dropna
    if DEBUG_VERBOSE:
        print(f"  [diagnostic] effective n_replicas: "
              f"prot_wt={len(prot_mean_wt_replica)}, nonprot_wt={len(nonprot_mean_wt_replica)}, "
              f"cr_prot={len(conflict_rate_prot_replica)}, cr_nonprot={len(conflict_rate_nonprot_replica)}")

    idle_mean, idle_margin = confidence_interval(idle_time_replica)
    over_mean, over_margin = confidence_interval(over_time_replica)
    overall_wt_mean, overall_wt_margin = confidence_interval(overall_patient_wt_replica)
    total_att_mean, total_att_margin = confidence_interval(total_attended_replica)

    prot_wt_mean, prot_wt_margin = confidence_interval(prot_mean_wt_replica)
    nonprot_wt_mean, nonprot_wt_margin = confidence_interval(nonprot_mean_wt_replica)

    cr_prot_mean, cr_prot_margin = confidence_interval(conflict_rate_prot_replica)
    cr_nonprot_mean, cr_nonprot_margin = confidence_interval(conflict_rate_nonprot_replica)

    # ========================================================
    # HYPOTHESIS TESTS — only for paired group comparisons.
    # Single metrics (idle, over, overall_wt, total_attended) get
    # descriptive stats only — there's no group-comparison test for them.
    # ========================================================
    tests_patient_wt = run_metric_tests(prot_mean_wt_replica, nonprot_mean_wt_replica, "PatientWT")
    tests_conflict = run_metric_tests(conflict_rate_prot_replica, conflict_rate_nonprot_replica, "ConflictRate")

    def _paired_row(prot_mean, prot_margin, nonprot_mean, nonprot_margin, tests_df):
        """Returns a flat dict summarising one threshold pair for a paired metric."""
        row = {
            "protected_threshold":     threshold_protected,
            "non_protected_threshold": threshold_no_protected,
            "replicas":                iterations,
            "inner_iterations":        NUM_ITERATIONS,
            "protected_ppv":           protected_ppv,
            "non_protected_ppv":       non_protected_ppv,
            "protected_npv":           protected_npv,
            "non_protected_npv":       non_protected_npv,
            "prot_mean":               prot_mean,
            "prot_ci_lower":           prot_mean - prot_margin,
            "prot_ci_upper":           prot_mean + prot_margin,
            "nonprot_mean":            nonprot_mean,
            "nonprot_ci_lower":        nonprot_mean - nonprot_margin,
            "nonprot_ci_upper":        nonprot_mean + nonprot_margin,
        }
        # Flatten hypothesis-test columns (one column per test row x field)
        for _, tr in tests_df.iterrows():
            prefix = f"{tr['metric']}_{tr['alternative']}"
            row[f"{prefix}_t_stat"]  = tr["t_stat"]
            row[f"{prefix}_p_value"] = tr["p_value"]
            row[f"{prefix}_welch"]   = tr["welch_correction"]
        return row

    def _single_row(mean_val, margin, series):
        """Returns a flat dict summarising one threshold pair for a single-group metric."""
        return {
            "protected_threshold":     threshold_protected,
            "non_protected_threshold": threshold_no_protected,
            "replicas":                iterations,
            "inner_iterations":        NUM_ITERATIONS,
            "protected_ppv":           protected_ppv,
            "non_protected_ppv":       non_protected_ppv,
            "protected_npv":           protected_npv,
            "non_protected_npv":       non_protected_npv,
            "mean":                    mean_val,
            "ci_lower":                mean_val - margin,
            "ci_upper":                mean_val + margin,
            "n_replicas":              len(series),
        }

    metric_rows["PatientWT"].append(
        _paired_row(prot_wt_mean, prot_wt_margin,
                    nonprot_wt_mean, nonprot_wt_margin,
                    tests_patient_wt))

    metric_rows["ConflictRate"].append(
        _paired_row(cr_prot_mean, cr_prot_margin,
                    cr_nonprot_mean, cr_nonprot_margin,
                    tests_conflict))

    metric_rows["IdleTime"].append(
        _single_row(idle_mean, idle_margin, idle_time_replica))

    metric_rows["OverTime"].append(
        _single_row(over_mean, over_margin, over_time_replica))

    metric_rows["OverallWT"].append(
        _single_row(overall_wt_mean, overall_wt_margin, overall_patient_wt_replica))

    metric_rows["TotalAttended"].append(
        _single_row(total_att_mean, total_att_margin, total_attended_replica))

    summary_row = {
        "rule": RULE_NAME,
        "protected_threshold": threshold_protected,
        "non_protected_threshold": threshold_no_protected,
        "replicas": iterations,
        "inner_iterations": NUM_ITERATIONS,
        "protected_ppv": protected_ppv,
        "non_protected_ppv": non_protected_ppv,
        "protected_npv": protected_npv,
        "non_protected_npv": non_protected_npv,
        "idle_time": idle_mean,
        "over_time": over_mean,
        "overall_patient_wt": overall_wt_mean,
        "total_attended": total_att_mean,
        "protected_patient_wt": prot_wt_mean,
        "non_protected_patient_wt": nonprot_wt_mean,
        "protected_conflict_rate": cr_prot_mean,
        "non_protected_conflict_rate": cr_nonprot_mean,


        "protected_anchors":          prot_anchor,
        "protected_flagged":          prot_flagged,
        "non_protected_anchors":      nonprot_anchor,
        "non_protected_flagged":      nonprot_flagged,
        "protected_flagged_pct":      prot_flagged_pct,
        "non_protected_flagged_pct":  nonprot_flagged_pct,
        "flagged_pct_asymmetry":      prot_flagged_pct - nonprot_flagged_pct,

    }

    results_summary_rows.append(summary_row)


# ============================================================
# #                       WRITE EXCEL                        #
# ============================================================

results_summary_df = pd.DataFrame(results_summary_rows)

base_dir = Path("simulation_outputs/excel_files/post_thesis_iterations")
base_dir.mkdir(parents=True, exist_ok=True)

summary_path = base_dir / f"{files_prefix}_{NUM_REPLICAS}_replicas_{NUM_ITERATIONS}_iter_summary.xlsx"

summary_path.unlink(missing_ok=True)

with pd.ExcelWriter(summary_path, engine="openpyxl") as writer:

    # Summary sheet — one row per threshold pair (unchanged)
    results_summary_df.to_excel(writer, sheet_name="Summary", index=False)
    autosize_excel_columns(writer, "Summary")

    # One sheet per metric — one row per threshold pair
    for metric_name, rows in metric_rows.items():
        df_metric = pd.DataFrame(rows)
        sheet_name = metric_name[:31]
        df_metric.to_excel(writer, sheet_name=sheet_name, index=False)
        autosize_excel_columns(writer, sheet_name)

print(f"[summary] Saved {len(results_summary_rows)} rows -> {summary_path}")

Simulating PT 0.25 & NPT 0.25 | [P PPV 0.2716] [NP PPV 0.5525] [P NPV 0.8935] [NP NPV 0.9109]
Ran Successfully Replica 1/30 in 4.72s
Ran Successfully Replica 2/30 in 4.67s
Ran Successfully Replica 3/30 in 4.70s
Ran Successfully Replica 4/30 in 4.73s
Ran Successfully Replica 5/30 in 4.49s
Ran Successfully Replica 6/30 in 4.76s
Ran Successfully Replica 7/30 in 4.54s
Ran Successfully Replica 8/30 in 4.69s
Ran Successfully Replica 9/30 in 4.59s
Ran Successfully Replica 10/30 in 4.73s
Ran Successfully Replica 11/30 in 4.53s
Ran Successfully Replica 12/30 in 4.57s
Ran Successfully Replica 13/30 in 4.45s
Ran Successfully Replica 14/30 in 4.62s
Ran Successfully Replica 15/30 in 4.61s
Ran Successfully Replica 16/30 in 4.83s
Ran Successfully Replica 17/30 in 4.70s
Ran Successfully Replica 18/30 in 4.59s
Ran Successfully Replica 19/30 in 4.62s
Ran Successfully Replica 20/30 in 4.36s
Ran Successfully Replica 21/30 in 4.25s
Ran Successfully Replica 22/30 in 4.32s
Ran Successfully Replica 23/30 in 4

In [ ]:
files_prefix = (
    f"{RULE_NAME}_overbooking{OVERBOOKING_LEVEL}_sample{TOTAL_PATIENT_POOL}_prot{PROTECTED_GROUP_PROPORTION:.2f}"
)

DEBUG_VERBOSE = False
model = None

for threshold_iter, thresholds in enumerate(THRESHOLDS):

    clear_output(wait=True)
    
    threshold_protected = thresholds[0]
    threshold_no_protected = thresholds[1]

    protected_ppv = ppv_npv_df[ppv_npv_df["threshold"] == threshold_protected]['protected_ppv'].values[0]
    non_protected_ppv = ppv_npv_df[ppv_npv_df["threshold"] == threshold_no_protected]['unprotected_ppv'].values[0]
    protected_npv = ppv_npv_df[ppv_npv_df["threshold"] == threshold_protected]['protected_npv'].values[0]
    non_protected_npv = ppv_npv_df[ppv_npv_df["threshold"] == threshold_no_protected]['unprotected_npv'].values[0]

    with open('patients_with_proba.pkl', 'rb') as f:
        patients_og = pickle.load(f) 
        
    all_measures = []
    convergence_values = []

    print(f"Simulating PT {threshold_protected} & NPT {threshold_no_protected} | [P PPV {protected_ppv:.4f}] [NP PPV {non_protected_ppv:.4f}] [P NPV {protected_npv:.4f}] [NP NPV {non_protected_npv:.4f}]")

    # region montecarlo
    convergence = False
    iterations = 0
    while not convergence and iterations < NUM_REPLICAS:
        start = time.time()
        iterations += 1

        patients = copy.deepcopy(patients_og)

        if DEBUG_VERBOSE:
            print(f"After deepcopy {iterations} iterations | {time.time():.4f}")

        created_appointments = []
        created_appointments = create_appointments(NUM_SERVERS, SIMULATION_DAYS, OPERATING_HOURS, SLOT_DURATION_MINS)

        # ── Sample + composition diagnostic ──────────────────────────────────
        sampled_random_patient_list = random_patient_sample(patients, TOTAL_PATIENT_POOL, PROTECTED_GROUP_PROPORTION, SIMULATION_DAYS)

        if iterations == 1:
            prot_in_sample    = [p for p in sampled_random_patient_list if p.protected]
            nonprot_in_sample = [p for p in sampled_random_patient_list if not p.protected]
            expected_high_proba_guaranteed = int(TOTAL_PATIENT_POOL * PROTECTED_GROUP_PROPORTION) // 10
            high_proba_prot_count = sum(1 for p in prot_in_sample if p.proba > 0.3)
            print(f"  [sample] Protected  n={len(prot_in_sample)}, "
                  f"mean proba={np.mean([p.proba for p in prot_in_sample]):.4f}, "
                  f"above th({threshold_protected})={sum(1 for p in prot_in_sample if p.proba > threshold_protected)}")
            print(f"  [sample] NonProt    n={len(nonprot_in_sample)}, "
                  f"mean proba={np.mean([p.proba for p in nonprot_in_sample]):.4f}, "
                  f"above th({threshold_no_protected})={sum(1 for p in nonprot_in_sample if p.proba > threshold_no_protected)}")
            print(f"  [sample] High-proba protected (>0.3): {high_proba_prot_count} "
                  f"(guaranteed minimum: {expected_high_proba_guaranteed})")

        if DEBUG_VERBOSE:
            print(f"IDs in random_patient_list: {[patient.id for patient in sampled_random_patient_list]}")

        # ── Rule assignment + pairing diagnostic ─────────────────────────────
        appointments_from_rule, num_refused = call_a_rule(
            sampled_random_patient_list, created_appointments,
            RULE_NAME, model,
            threshold_protected, threshold_no_protected,
            overbooking_level=OVERBOOKING_LEVEL
        )

        paired_prot_prot       = 0
        paired_nonprot_nonprot = 0
        paired_mixed           = 0
        total_double_slots     = 0
        patient_map = {p.id: p for p in sampled_random_patient_list}

        for server in appointments_from_rule:
            for day in server:
                for slot in day:
                    if isinstance(slot, list):
                        occupants = [pid for pid in slot if pid is not None]
                        if len(occupants) == 2:
                            total_double_slots += 1
                            p1 = patient_map[occupants[0]]
                            p2 = patient_map[occupants[1]]
                            if p1.protected and p2.protected:
                                paired_prot_prot += 1
                            elif (not p1.protected) and (not p2.protected):
                                paired_nonprot_nonprot += 1
                            else:
                                paired_mixed += 1

        if iterations == 1 and threshold_iter == 0:
            print(f"  [pairing] total double-booked slots : {total_double_slots}")
            print(f"  [pairing] Prot-Prot                : {paired_prot_prot}")
            print(f"  [pairing] NonProt-NonProt           : {paired_nonprot_nonprot}")
            print(f"  [pairing] Mixed                     : {paired_mixed}")

        if DEBUG_VERBOSE:
            print(f"IDs RandomPatientList after calling rule: {[patient.id for patient in sampled_random_patient_list]}")
            print(f"\nNumber of patients not assigned {num_refused}")

        # ── Inner simulation loop ─────────────────────────────────────────────
        for iter in range(NUM_ITERATIONS):

            appointments = copy.deepcopy(appointments_from_rule) 
            random_patient_list = copy.deepcopy(sampled_random_patient_list)

            attendance_random_patient_list = establish_attendance(
                random_patient_list,
                protected_ppv, non_protected_ppv,
                protected_npv, non_protected_npv,
                threshold_protected, threshold_no_protected
            )

            clinic = Clinic(attendance_random_patient_list, appointments, SLOT_DURATION_MINS)
            clinic.simulation()
            measures = clinic.get_measures()

            if DEBUG_VERBOSE:
                print(f"Appointments after scheduling simulation: {appointments}")
                for server in appointments:
                    for i, day in enumerate(server):
                        print(f"\nD{i}")
                        for j, slot in enumerate(day):
                            for appoint_test in slot:
                                print(f"P{appoint_test}|ID{random_patient_list[appoint_test].id}|{random_patient_list[appoint_test].num_slot}", end=", ")
                print("measures:", measures)

            all_measures.append(measures)

            del appointments
            del random_patient_list
            del attendance_random_patient_list
            del clinic

        # -- Convergence check on per-replica aggregates ---------------------------
        # `all_measures` contains `iterations * NUM_ITERATIONS` rows, where
        # the inner `NUM_ITERATIONS` rows within each replica share the same
        # schedule (only attendance realization differs). Those rows are NOT
        # independent. For an honest convergence criterion, aggregate to one record
        # per replica, then check margin/mean on the aggregated series.
        if iterations >= 15:
            _tmp_df = pd.DataFrame(all_measures).copy()
            _tmp_df["replica_id"] = np.repeat(
                np.arange(iterations), NUM_ITERATIONS
            )[:len(_tmp_df)]

            # Aggregate inner-iteration records into one per replica (mean of the
            # attendance-realization distribution for that schedule).
            _replica_records = (
                _tmp_df
                .drop(columns=["idle_time_server", "over_time"], errors="ignore")
                .groupby("replica_id")
                .mean(numeric_only=True)
                .to_dict(orient="records")
            )

            convergence, diff = check_convergence_mean(
                _replica_records,
                converge_mean=0.02,   # tight: need to resolve ~2% effect sizes
                verbose=DEBUG_VERBOSE,
            )
            convergence_values.append(diff)

            if DEBUG_VERBOSE:
                print(f"  [convergence] iter {iterations}: worst margin/mean = "
                    f"{diff:.4f} ({'CONVERGED' if convergence else 'continuing'})")

        end = time.time()
        print(f"Ran Successfully Replica {iterations}/{NUM_REPLICAS} in {end-start:.2f}s")

        del created_appointments
        del sampled_random_patient_list
        del patients

    print(f"* * * Ran Successfully All Replicas!!!\n")
    # endregion

    # region metric_calculations
    measures_df = pd.DataFrame(all_measures)
    summary_measures = get_margin_errors(all_measures)
    summary_measures_df = calculate_summary(measures_df)

    idle_time_mean        = summary_measures_df['mean'][0] / 7
    idle_time_lower_bound = summary_measures_df['confidence_interval'][0][0] / 7
    idle_time_upper_bound = summary_measures_df['confidence_interval'][0][1] / 7

    over_time_mean        = summary_measures_df['mean'][1] / 7
    over_time_lower_bound = summary_measures_df['confidence_interval'][1][0] / 7
    over_time_upper_bound = summary_measures_df['confidence_interval'][1][1] / 7

    no_shows_mean         = summary_measures_df['mean'][2] / 7
    no_shows_lower_bound  = summary_measures_df['confidence_interval'][2][0] / 7
    no_shows_upper_bound  = summary_measures_df['confidence_interval'][2][1] / 7

    patient_wt_row         = summary_measures_df[summary_measures_df["column"] == "patient_waiting_time"].iloc[0]
    patient_wt_mean        = patient_wt_row["mean"]
    patient_wt_lower_bound = patient_wt_row["confidence_interval"][0]
    patient_wt_upper_bound = patient_wt_row["confidence_interval"][1]

    # -- Per-replica aggregation for hypothesis-test series --------------------
    # Each replica runs one schedule and `NUM_ITERATIONS` attendance
    # realizations on that schedule. The realizations share the same rule
    # decisions and are therefore correlated. For valid statistical inference,
    # aggregate to the replica level first, then treat replicas as the
    # independent sample.
    measures_df["replica_id"] = np.repeat(
        np.arange(iterations), NUM_ITERATIONS
    )[:len(measures_df)]

    replica_grouped = (
        measures_df
        .groupby("replica_id")
        .agg(
            prot_wt_total    = ("protected_overbooked_waiting_time",     "sum"),
            nonprot_wt_total = ("non_protected_overbooked_waiting_time", "sum"),
            prot_ob_n        = ("protected_overbooked_patients",         "sum"),
            nonprot_ob_n     = ("non_protected_overbooked_patients",     "sum"),
            prot_attend      = ("protected_assistance",                  "sum"),
            nonprot_attend   = ("non_protected_assistance",              "sum"),
            conflict_prot    = ("conflict_slots_protected",              "sum"),
            conflict_nonprot = ("conflict_slots_non_protected",          "sum"),
        )
        .reset_index()
    )

    # Per-replica mean overbooked waiting time (stable denominator across the
    # replica's 30 inner realizations). Drop replicas with zero overbooked
    # patients — they carry no information for this metric.
    ob_prot_wt = (
        replica_grouped["prot_wt_total"] /
        replica_grouped["prot_ob_n"].replace(0, np.nan)
    ).dropna()
    ob_non_prot_wt = (
        replica_grouped["nonprot_wt_total"] /
        replica_grouped["nonprot_ob_n"].replace(0, np.nan)
    ).dropna()

    # Per-replica conflict rate (cleaner primary metric — unbounded by slot_time
    # and directly measures exposure to conflict events).
    conflict_rate_prot_per_replica = (
        replica_grouped["conflict_prot"] /
        replica_grouped["prot_attend"].replace(0, np.nan)
    ).dropna()
    conflict_rate_nonprot_per_replica = (
        replica_grouped["conflict_nonprot"] /
        replica_grouped["nonprot_attend"].replace(0, np.nan)
    ).dropna()

    protected_wt_mean,     protected_wt_margin     = confidence_interval(ob_prot_wt)
    non_protected_wt_mean, non_protected_wt_margin = confidence_interval(ob_non_prot_wt)

    total_attended_mean,  total_attended_margin  = confidence_interval(measures_df["total_attended_patients"])

    conflict_prot_mean,     conflict_prot_margin     = confidence_interval(measures_df["conflict_slots_protected"])
    conflict_nonprot_mean,  conflict_nonprot_margin  = confidence_interval(measures_df["conflict_slots_non_protected"])

    conflict_rate_prot_mean,    conflict_rate_prot_margin    = confidence_interval(measures_df["conflict_rate_protected"])
    conflict_rate_nonprot_mean, conflict_rate_nonprot_margin = confidence_interval(measures_df["conflict_rate_non_protected"])

    one_sided_correction_df     = ttest(ob_prot_wt, ob_non_prot_wt, correction=True,  alternative='greater',   confidence=0.95)
    one_sided_non_correction_df = ttest(ob_prot_wt, ob_non_prot_wt, correction=False, alternative='greater',   confidence=0.95)
    two_sided_correction_df     = ttest(ob_prot_wt, ob_non_prot_wt, correction=True,  alternative='two-sided', confidence=0.95)
    two_sided_non_correction_df = ttest(ob_prot_wt, ob_non_prot_wt, correction=False, alternative='two-sided', confidence=0.95)

    one_sided_correction_p_val     = one_sided_correction_df["p_val"].values[0]
    one_sided_correction_t         = one_sided_correction_df["T"].values[0]
    one_sided_non_correction_p_val = one_sided_non_correction_df["p_val"].values[0]
    one_sided_non_correction_t     = one_sided_non_correction_df["T"].values[0]
    two_sided_correction_p_val     = two_sided_correction_df["p_val"].values[0]
    two_sided_correction_t         = two_sided_correction_df["T"].values[0]
    two_sided_non_correction_p_val = two_sided_non_correction_df["p_val"].values[0]
    two_sided_non_correction_t     = two_sided_non_correction_df["T"].values[0]

    # -- Conflict-rate t-tests (primary mechanistic metric) --------------------
    cr_one_sided_correction_df = ttest(
        conflict_rate_prot_per_replica, conflict_rate_nonprot_per_replica,
        correction=True, alternative='greater', confidence=0.95
    )
    cr_two_sided_correction_df = ttest(
        conflict_rate_prot_per_replica, conflict_rate_nonprot_per_replica,
        correction=True, alternative='two-sided', confidence=0.95
    )
    cr_one_sided_non_correction_df = ttest(
        conflict_rate_prot_per_replica, conflict_rate_nonprot_per_replica,
        correction=False, alternative='greater', confidence=0.95
    )
    cr_two_sided_non_correction_df = ttest(
        conflict_rate_prot_per_replica, conflict_rate_nonprot_per_replica,
        correction=False, alternative='two-sided', confidence=0.95
    )

    cr_one_sided_correction_p_val     = cr_one_sided_correction_df["p_val"].values[0]
    cr_one_sided_correction_t         = cr_one_sided_correction_df["T"].values[0]
    cr_two_sided_correction_p_val     = cr_two_sided_correction_df["p_val"].values[0]
    cr_two_sided_correction_t         = cr_two_sided_correction_df["T"].values[0]
    cr_one_sided_non_correction_p_val = cr_one_sided_non_correction_df["p_val"].values[0]
    cr_one_sided_non_correction_t     = cr_one_sided_non_correction_df["T"].values[0]
    cr_two_sided_non_correction_p_val = cr_two_sided_non_correction_df["p_val"].values[0]
    cr_two_sided_non_correction_t     = cr_two_sided_non_correction_df["T"].values[0]

    # region summary_df_and_excel
    summary_row = {
        # ══════════════════════════════════════════════════════════════════════
        # 1. EXPERIMENT IDENTITY — what scenario is this row?
        # ══════════════════════════════════════════════════════════════════════
        "rule":                              f"{RULE_NAME.replace('_', ' ').title()} {OVERBOOKING_LEVEL}",
        "protected_proportion":              PROTECTED_GROUP_PROPORTION,
        "unprotected_proportion":            1 - PROTECTED_GROUP_PROPORTION,
        "protected_threshold":               threshold_protected,
        "unprotected_threshold":             threshold_no_protected,
        "iterations":                        f"{iterations} replicas | {NUM_ITERATIONS} iter",

        # ══════════════════════════════════════════════════════════════════════
        # 2. BIAS SOURCE — the model-calibration asymmetry being tested
        # ══════════════════════════════════════════════════════════════════════
        "protected_ppv":                     protected_ppv,
        "unprotected_ppv":                   non_protected_ppv,
        "protected_npv":                     protected_npv,
        "unprotected_npv":                   non_protected_npv,

        # ══════════════════════════════════════════════════════════════════════
        # 3. PRIMARY OUTCOME — conflict rate (cleanest mechanistic signal)
        # ══════════════════════════════════════════════════════════════════════
        "conflict_rate_protected_mean":      conflict_rate_prot_mean,
        "conflict_rate_non_protected_mean":  conflict_rate_nonprot_mean,
        "conflict_rate_protected_lower":     conflict_rate_prot_mean    - conflict_rate_prot_margin,
        "conflict_rate_non_protected_lower": conflict_rate_nonprot_mean - conflict_rate_nonprot_margin,
        "conflict_rate_protected_upper":     conflict_rate_prot_mean    + conflict_rate_prot_margin,
        "conflict_rate_non_protected_upper": conflict_rate_nonprot_mean + conflict_rate_nonprot_margin,

        # Conflict-rate hypothesis tests (primary)
        "cr_one_sided_pval_corr":            cr_one_sided_correction_p_val,
        "cr_two_sided_pval_corr":            cr_two_sided_correction_p_val,
        "cr_one_sided_t_corr":               cr_one_sided_correction_t,
        "cr_two_sided_t_corr":               cr_two_sided_correction_t,
        "cr_one_sided_pval_non_corr":        cr_one_sided_non_correction_p_val,
        "cr_two_sided_pval_non_corr":        cr_two_sided_non_correction_p_val,
        "cr_one_sided_t_non_corr":           cr_one_sided_non_correction_t,
        "cr_two_sided_t_non_corr":           cr_two_sided_non_correction_t,

        # ══════════════════════════════════════════════════════════════════════
        # 4. SECONDARY OUTCOME — overbooked waiting time (bounded by slot_time)
        # ══════════════════════════════════════════════════════════════════════
        "ob_protected_wt_mean":              protected_wt_mean,
        "ob_non_protected_wt_mean":          non_protected_wt_mean,
        "ob_protected_wt_lower":             protected_wt_mean     - protected_wt_margin,
        "ob_non_protected_wt_lower":         non_protected_wt_mean - non_protected_wt_margin,
        "ob_protected_wt_upper":             protected_wt_mean     + protected_wt_margin,
        "ob_non_protected_wt_upper":         non_protected_wt_mean + non_protected_wt_margin,

        # Overbooked-WT hypothesis tests (secondary)
        "one_sided_pval_corr":               one_sided_correction_p_val,
        "two_sided_pval_corr":               two_sided_correction_p_val,
        "one_sided_t_corr":                  one_sided_correction_t,
        "two_sided_t_corr":                  two_sided_correction_t,
        "one_sided_pval_non_corr":           one_sided_non_correction_p_val,
        "two_sided_pval_non_corr":           two_sided_non_correction_p_val,
        "one_sided_t_non_corr":              one_sided_non_correction_t,
        "two_sided_t_non_corr":              two_sided_non_correction_t,

        # ══════════════════════════════════════════════════════════════════════
        # 5. MECHANISM DIAGNOSTICS — how the effect is produced
        # ══════════════════════════════════════════════════════════════════════
        # Conflict-slot event counts (raw numerator of conflict rate)
        "conflict_slots_protected_mean":      conflict_prot_mean,
        "conflict_slots_non_protected_mean":  conflict_nonprot_mean,
        "conflict_slots_protected_lower":     conflict_prot_mean    - conflict_prot_margin,
        "conflict_slots_non_protected_lower": conflict_nonprot_mean - conflict_nonprot_margin,
        "conflict_slots_protected_upper":     conflict_prot_mean    + conflict_prot_margin,
        "conflict_slots_non_protected_upper": conflict_nonprot_mean + conflict_nonprot_margin,

        # Pairing composition — whose slots conflict with whose?
        "total_double_slots":                total_double_slots,
        "paired_prot_prot":                  paired_prot_prot,
        "paired_nonprot_nonprot":            paired_nonprot_nonprot,
        "paired_mixed":                      paired_mixed,

        # ══════════════════════════════════════════════════════════════════════
        # 6. OPERATIONAL CONTEXT — clinic-level realism sanity checks
        # ══════════════════════════════════════════════════════════════════════
        "overall_patient_wt_mean":           patient_wt_mean,
        "overall_patient_wt_lower":          patient_wt_lower_bound,
        "overall_patient_wt_upper":          patient_wt_upper_bound,
        "total_attended_mean":               total_attended_mean,
        "total_attended_lower":              total_attended_mean - total_attended_margin,
        "total_attended_upper":              total_attended_mean + total_attended_margin,
        "idle_time_mean":                    idle_time_mean,
        "idle_time_lower":                   idle_time_lower_bound,
        "idle_time_upper":                   idle_time_upper_bound,
        "over_time_mean":                    over_time_mean,
        "over_time_lower":                   over_time_lower_bound,
        "over_time_upper":                   over_time_upper_bound,
    }

    if threshold_iter == 0:
        results_summary_rows = []
    results_summary_rows.append(summary_row)

    results_summary_df = pd.DataFrame(results_summary_rows)

    base_dir = Path("simulation_outputs/excel_files/post_thesis_iterations")
    base_dir.mkdir(parents=True, exist_ok=True)
    summary_path = base_dir / f"{files_prefix}_{NUM_REPLICAS}_replicas_{NUM_ITERATIONS}_iter_summary.xlsx"

    with pd.ExcelWriter(summary_path, engine="openpyxl") as writer:
        results_summary_df.to_excel(writer, index=False, sheet_name="Summary")

        worksheet = writer.sheets["Summary"]
        for col_cells in worksheet.columns:
            max_len = max(
                len(str(cell.value)) if cell.value is not None else 0
                for cell in col_cells
            )
            worksheet.column_dimensions[col_cells[0].column_letter].width = max(max_len + 2, 12)

    print(f"  [summary] Saved {len(results_summary_rows)} threshold pair(s) → {summary_path}")